In [19]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import timm

In [20]:
images_folder = '/kaggle/input/competitions/ocr-showdown-decode-12-digit-roll-numbers/roll_number_dataset2'
labels = pd.read_csv("/kaggle/input/competitions/ocr-showdown-decode-12-digit-roll-numbers/roll_number_dataset2/labels.csv")
labels

,image_path,label
0,images/134712536281_0.png,134712536281
1,images/057665706184_1.png,57665706184
2,images/236070779846_2.png,236070779846
3,images/143539849576_3.png,143539849576
4,images/717365979832_4.png,717365979832
...,...,...
9995,images/513992498632_9995.png,513992498632
9996,images/991413267472_9996.png,991413267472
9997,images/076985176225_9997.png,76985176225
9998,images/842951915653_9998.png,842951915653


In [21]:
VOCAB = {str(i): i for i in range(10)}
SOS_TOKEN = 10
EOS_TOKEN = 11

In [22]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class OCRDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx, 0])
        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path):
            img_path += '.jpg'
            
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(self.df.iloc[idx, 1]).strip()
        tokens = [SOS_TOKEN] + [VOCAB[c] for c in text if c in VOCAB] + [EOS_TOKEN]
        
        decoder_input = torch.tensor(tokens[:-1], dtype=torch.long)
        target = torch.tensor(tokens[1:], dtype=torch.long)

        return image, decoder_input, target

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = OCRDataset(labels, images_folder, transform)
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    images, dec_inputs, targets = zip(*batch)
    
    images = torch.stack(images)
    
    dec_inputs = pad_sequence(dec_inputs, batch_first=True, padding_value=EOS_TOKEN)
    targets = pad_sequence(targets, batch_first=True, padding_value=EOS_TOKEN)
    
    return images, dec_inputs, targets

dataloader = DataLoader(
    dataset, 
    batch_size=64,
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True 
)

In [26]:
class visionencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=True,
            num_classes=0
        )
        self.proj = nn.Linear(192, 768)
    def forward(self, x):
        memory = self.vit.forward_features(x)
        memory = memory[:, 1:]
        return self.proj(memory)


In [29]:
import torch
import torch.nn as nn
import torchvision.models as models

class OCRModelResNet(nn.Module):
    def __init__(self, num_classes=12, hidden_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) 
        
        self.pool = nn.AdaptiveAvgPool2d((1, 32)) 
        
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        
        self.token_emb = nn.Embedding(num_classes, hidden_dim * 2)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim * 2,
            nhead=8,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def generate_mask(self, length, device):
        mask = torch.triu(torch.ones(length, length, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, image, decoder_input):
        features = self.backbone(image)
        features = self.pool(features).squeeze(2) 
        features = features.permute(0, 2, 1)
        
        memory, _ = self.lstm(features)
        
        tgt = self.token_emb(decoder_input) 
        tgt_mask = self.generate_mask(tgt.shape[1], tgt.device)
        
        out = self.decoder(tgt=tgt, memory=memory, tgt_mask=tgt_mask)
        return self.fc(out)

In [30]:
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = OCRModelResNet().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=EOS_TOKEN)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda')

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for images, dec_inputs, targets in dataloader:
        images = images.to(device, non_blocking=True)
        dec_inputs = dec_inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            logits = model(images, dec_inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(dataloader):.4f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s] 


Epoch 1/10, Loss: 1.8079
Epoch 2/10, Loss: 0.7970
Epoch 3/10, Loss: 0.4467
Epoch 4/10, Loss: 0.3188
Epoch 5/10, Loss: 0.2485
Epoch 6/10, Loss: 0.2094
Epoch 7/10, Loss: 0.1848
Epoch 8/10, Loss: 0.1674
Epoch 9/10, Loss: 0.1556
Epoch 10/10, Loss: 0.1397


In [34]:
def predict_single_image(model, image_path, transform, device):
    model.eval()
    
    # 1. Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device) 
    
    decoder_input = torch.tensor([[SOS_TOKEN]], dtype=torch.long, device=device)
    
    with torch.no_grad():
        features = model.backbone(image)
        features = model.pool(features).squeeze(2).permute(0, 2, 1)
        memory, _ = model.lstm(features)
        
        for _ in range(12):
            tgt = model.token_emb(decoder_input)
            tgt_mask = model.generate_mask(tgt.shape[1], tgt.device)
            
            out = model.decoder(tgt=tgt, memory=memory, tgt_mask=tgt_mask)
            logits = model.fc(out)
            
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            
            if next_token.item() == EOS_TOKEN:
                break
                
            decoder_input = torch.cat([decoder_input, next_token], dim=1)
            
    predicted_tokens = decoder_input[0, 1:].cpu().tolist() # Skip initial SOS_TOKEN
    predicted_text = "".join([str(tok) for tok in predicted_tokens if tok < 10])
    
    return predicted_text

In [37]:
image_file = "/kaggle/input/competitions/ocr-showdown-decode-12-digit-roll-numbers/roll_number_dataset2/images/002918290252_5675.png"

predicted_roll = predict_single_image(model, image_file, transform, device)
print(f"Predicted Roll Number: {predicted_roll}")

Predicted Roll Number: 291829025252
